**Step 3: Build the Name-Equivalence Dictionaries**

This notebook builds the three dictionaries of equivalent names that the experiments compare. A dictionary is a set of groups, each holding spellings of one name in Hebrew and Latin script (Rivka, רבקה, Rebeca, Rifke). It reads the hand-written dictionary `./inputs/d1_curated.csv` and `./outputs/step_1_persons.csv`, and builds each dictionary once:

1. **D1, curated**: the 105 hand-written given-name groups of `./inputs/d1_curated.csv`
2. **D2, learned from the archive**: pairs of a Latin and a Hebrew token of the same field recorded on the same person, kept when at least two training persons share the pair, and joined into groups when pairs share a spelling. Only training persons are used, so that the people searched for in validation and test never teach the dictionary their own names
3. **D3, both**: D1 and D2 together, with groups that share a spelling merged

It outputs `./outputs/step_3_dictionary_d1.csv`, `./outputs/step_3_dictionary_d2.csv` and `./outputs/step_3_dictionary_d3.csv`: one row per spelling, with its group, script, normalised form and display spelling.

**1. Configuration and Helpers**

This cell defines the settings, reads the inputs, and copies the normalisation of step 1, which the curated dictionary's spellings need.

1. `merge_groups`: joins groups that share a spelling (the connected components of the spellings), used for D2 and D3
2. `dictionary_rows`: turns groups into the rows of a dictionary file

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# Configuration and helpers  —  inputs, normalisation and group merging
#
# A spelling is identified by its script and normalised form, e.g.
# ("latin", "rebeca"); the display spelling is what an embedding reads.
# ─────────────────────────────────────────────────────────────────────────────
import csv
import json
import re
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path

# ═════════════════════════════════════════════════════════════════════════════
# GLOBAL CONFIG
# ═════════════════════════════════════════════════════════════════════════════
D1_SOURCE   = Path("inputs/d1_curated.csv")   # the hand-written dictionary
OUTPUTS_DIR = Path("outputs")                 # every step's outputs
MIN_SUPPORT = 2                               # training persons a D2 pair needs
RULE        = "─" * 72

FINAL_TO_MEDIAL   = str.maketrans("ךםןףץ", "כמנפצ")     # final letters → medial forms
HEBREW_LETTERS    = "\N{HEBREW LETTER ALEF}-\N{HEBREW LETTER TAV}"            # a regex range
HEBREW_MARKS      = re.compile("[\N{HEBREW ACCENT ETNAHTA}-\N{HEBREW POINT QAMATS QATAN}]")
HEBREW_NON_LETTER = re.compile(f"[^{HEBREW_LETTERS}]")   # anything but a letter


def normalise(text: str, script: str) -> str:
    """Normalise one spelling in its script, as step 1 does."""
    if script == "latin":
        decomposed = unicodedata.normalize("NFKD", text)
        return "".join(c for c in decomposed
                       if not unicodedata.combining(c) and c.isalpha()).lower()
    text = HEBREW_MARKS.sub("", unicodedata.normalize("NFKD", text))
    return HEBREW_NON_LETTER.sub("", text.translate(FINAL_TO_MEDIAL))


def merge_groups(groups: list[set[tuple]]) -> list[set[tuple]]:
    """Join groups that share a spelling, until no two groups overlap."""
    merged = []
    for group in groups:
        group = set(group)
        overlapping = [m for m in merged if m & group]
        for other in overlapping:
            group |= other
            merged.remove(other)
        merged.append(group)
    return merged


def dictionary_rows(name: str, groups: list[set[tuple]], displays: dict,
                    sources: dict) -> list[dict]:
    """Return the rows of one dictionary file, one per spelling.

    Args:
        name: the dictionary's name, used in its group ids (`D2_7`).
        groups: the groups, each a set of (script, norm) spellings.
        displays: the display spelling of every (script, norm).
        sources: where each spelling comes from (`d1`, `d2` or `d1+d2`).
    """
    ordered = sorted(groups, key=sorted)
    return [{"group_id": f"{name}_{number}", "script": script, "norm": norm,
             "display": displays[(script, norm)], "source": sources[(script, norm)]}
            for number, group in enumerate(ordered, start=1) for script, norm in sorted(group)]


def write_csv(path: Path, rows: list[dict]) -> None:
    """Write `rows` (dictionaries with the same keys) to the CSV file `path`."""
    with path.open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0]), lineterminator="\n")
        writer.writeheader()
        writer.writerows(rows)


with D1_SOURCE.open(encoding="utf-8", newline="") as _file:
    d1_source = list(csv.DictReader(_file))
with (OUTPUTS_DIR / "step_1_persons.csv").open(encoding="utf-8", newline="") as _file:
    train_persons = [json.loads(p["tokens"]) for p in csv.DictReader(_file)
                     if p["split"] == "train"]
print(f"✓ Configuration and helpers loaded ({len(d1_source)} curated spellings, "
      f"{len(train_persons)} training persons).")

✓ Configuration and helpers loaded (564 curated spellings, 381 training persons).


**2. The Three Dictionaries**

This cell builds D1 from `d1_source`, D2 from `train_persons` and D3 from both, and writes each to `OUTPUTS_DIR`. A D2 pair counts each training person once, however many times the pair appears on their stone.

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# The three dictionaries  —  curated, learned from the archive, and both
# ─────────────────────────────────────────────────────────────────────────────
if "train_persons" not in globals():
    raise RuntimeError("Run the 'Configuration and Helpers' cell first.")

display, source = {}, {}

# ── D1: the curated groups ────────────────────────────────────────────────
_d1 = defaultdict(set)
for _row in d1_source:
    _key = (_row["script"], normalise(_row["form"], _row["script"]))
    _d1[_row["group_id"]].add(_key)
    display.setdefault(_key, _row["form"])
    source[_key] = "d1"
d1_groups = list(_d1.values())

# ── D2: Latin–Hebrew pairs shared by at least MIN_SUPPORT training persons ──
_support, _spellings = Counter(), defaultdict(Counter)
for _tokens in train_persons:
    for _token in _tokens:
        _spellings[(_token["script"], _token["norm"])][_token["display"]] += 1
    _pairs = {(("latin", a["norm"]), ("hebrew", b["norm"]))
              for a in _tokens for b in _tokens
              if a["field"] == b["field"] and a["script"] == "latin" and b["script"] == "hebrew"}
    _support.update(_pairs)
d2_groups = merge_groups([set(pair) for pair, count in _support.items() if count >= MIN_SUPPORT])
for _group in d2_groups:
    for _key in _group:
        display[_key] = _spellings[_key].most_common(1)[0][0]
        source[_key] = "d1+d2" if source.get(_key) == "d1" else "d2"

# ── D3: both, merged where they share a spelling ─────────────────────────────
d3_groups = merge_groups(d1_groups + d2_groups)
for _name, _groups in (("D1", d1_groups), ("D2", d2_groups), ("D3", d3_groups)):
    write_csv(OUTPUTS_DIR / f"step_3_dictionary_{_name.lower()}.csv",
              dictionary_rows(_name, _groups, display, source))

print(f"{RULE}\nDictionaries\n{RULE}")
for _name, _groups in (("D1", d1_groups), ("D2", d2_groups), ("D3", d3_groups)):
    _example = sorted(max(_groups, key=len))
    print(f"  {_name}  {len(_groups):>4} groups  {sum(len(g) for g in _groups):>5} spellings   "
          f"largest: {', '.join(display[k] for k in _example[:6])}")
print(f"{RULE}\n  {sum(count >= MIN_SUPPORT for count in _support.values())} D2 pairs from "
      f"{len(_support)} distinct Latin–Hebrew pairs")

────────────────────────────────────────────────────────────────────────
Dictionaries
────────────────────────────────────────────────────────────────────────
  D1   105 groups    564 spellings   largest: יאנקל, יעקב, קאפל, Iaacov, Iankel, Jacob
  D2    41 groups     96 spellings   largest: חיה, חנה, שיינדל, שרה, Ana, Clara
  D3   123 groups    622 spellings   largest: הענע, חיה, חייקע, חנה, יפה, שיינדל
────────────────────────────────────────────────────────────────────────
  55 D2 pairs from 462 distinct Latin–Hebrew pairs


**3. Consistency Check**

This cell reads the three files back and checks that every D1 and D2 spelling is in D3, and that no spelling belongs to two D3 groups.

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# Consistency check  —  D3 holds D1 and D2, each spelling once
# ─────────────────────────────────────────────────────────────────────────────
if "d3_groups" not in globals():
    raise RuntimeError("Run every cell above first.")

_stored = {}
for _name in ("d1", "d2", "d3"):
    with (OUTPUTS_DIR / f"step_3_dictionary_{_name}.csv").open(encoding="utf-8") as _file:
        _stored[_name] = [(r["script"], r["norm"]) for r in csv.DictReader(_file)]
_problems = [] if set(_stored["d1"]) | set(_stored["d2"]) == set(_stored["d3"]) else [
    "D3 does not hold exactly the spellings of D1 and D2"]
_problems += [f"{k} is in {n} D3 groups" for k, n in Counter(_stored["d3"]).items() if n > 1]
print(f"{RULE}\nConsistency: {OUTPUTS_DIR}/step_3_dictionary_*.csv\n{RULE}")
for _problem in _problems[:10]:
    print(f"  !! {_problem}")
print(f"  {'OK' if not _problems else '!!'} D1 {len(_stored['d1'])}, D2 {len(_stored['d2'])}, "
      f"D3 {len(_stored['d3'])} spellings")
print(f"\n✓ Dictionaries complete (D1 {len(d1_groups)}, D2 {len(d2_groups)} and "
      f"D3 {len(d3_groups)} groups).")
print(f"  consistent: {not _problems}")

────────────────────────────────────────────────────────────────────────
Consistency: outputs/step_3_dictionary_*.csv
────────────────────────────────────────────────────────────────────────
  OK D1 564, D2 96, D3 622 spellings

✓ Dictionaries complete (D1 105, D2 41 and D3 123 groups).
  consistent: True
